# Ordinary Least Squares Experiment

This notebook explores the ordinary least squares (OLS) solution for Runge function. We evaluate the solution using the mean squared error (MSE) and R2 function. 

First we import all that is necesarry.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data, runge_function
from fys_stk4155_p1.metrics import mean_squared_error, r2_score
from fys_stk4155_p1.regression.ordinary_least_squares import OLS

We can then generate and visualize the data.

In [ ]:
# Generate data
x, y = generate_runge_data(n=100, noise_std=0.1, seed=42)


x_plot = np.linspace(-1, 1, 500)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge's function")
ax.scatter(x, y, s=15, alpha=0.6, label=r"data, $\sigma = 0.1$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Runge function: noisy samples vs. ground truth")
ax.legend()
fig.tight_layout()
plt.show()

## Fit OLS across polynomial degree

For each degree $d$ we build the design matrix $[x, x^2, \dots, x^d]$ — no intercept
column (`intercept=False`), since `OLS`/`LinearModel` never adds one implicitly; see
`fys_stk4155_p1.regression.base`. To handle the intercept term $\theta_0$ without an
intercept column, and to avoid a `StandardScaler` dividing a constant column by a
standard deviation of zero, we follow the same convention already used by
`Ridge`/`shrinkage.py` elsewhere in this package (standardized features, centered
target, `fit_intercept_column=False`):

1. Fit a `StandardScaler` on the training split's columns and apply it to both train
   and test splits (fitting on train only avoids leaking test-set statistics into the
   transform).
2. Center `y` by subtracting the training mean, `y_mean = y_train.mean()`, and fit
   OLS on `(X_train_scaled, y_train - y_mean)`.
3. Because standardized features have zero mean, the true intercept is exactly
   `y_mean` — recovered by adding it back at prediction time
   (`y_pred = X_scaled @ theta + y_mean`) rather than by fitting it as an extra
   column.

With this layout `theta[i]` corresponds to the standardized $x^{i+1}$ term
(`theta[0]` is the $x^1$ coefficient, not an intercept), and the intercept is
returned separately.

> **TODO:** Part (a) also asks for a critical discussion of *why and how* we scaled
> the data. Write that discussion here once the design choices above are finalized
> (see Section 3.13 of the lecture notes, the Tuesday session of week 35 and the 
> pertinent notebook for week 35).

The fitting logic (design matrix → scale → OLS → metrics) is reused below for the
dataset-size sweep, so it's wrapped in a small helper function rather than repeated.

In [ ]:
def fit_polynomial_ols_by_degree(
    x: np.ndarray,
    y: np.ndarray,
    degrees: range,
    test_size: float = 0.2,
    seed: int = 42,
) -> dict:
    """Fit OLS on polynomial features of x, for each degree in `degrees`.

    Builds the design matrix [x, x^2, ..., x^max(degrees)] once (no intercept
    column), then for each degree slices out the first `degree` columns
    (x^1..x^degree), standardizes them with a scaler fit on the training split
    only, and fits OLS on the standardized features against the centered target
    `y_train - y_mean`. The intercept isn't a fitted column: since standardized
    features have zero mean, it's exactly `y_mean`, added back at prediction time.

    Args:
        x: x-coordinates, shape (n,).
        y: targets, shape (n,).
        degrees: polynomial degrees to fit, e.g. range(1, 16).
        test_size: fraction of samples held out for testing.
        seed: seed for the train/test split, for reproducibility.

    Returns:
        Dict with keys "degrees", "intercept" (scalar `y_train.mean()`, identical
        across degrees since it only depends on the split), "weights" (list of
        theta arrays, one per degree; theta[i] is the coefficient of the
        standardized x^(i+1) term), "mse_train", "mse_test", "r2_train", "r2_test"
        (arrays aligned with "degrees").
    """
    degrees_arr = np.array(list(degrees))
    max_degree = int(degrees_arr.max())

    X_full = univariate_polynomial_design_matrix(x=x, degree=max_degree, intercept=False)

    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X_full, y, test_size=test_size, random_state=seed
    )

    y_mean = y_train.mean()
    y_train_centered = y_train - y_mean

    weights = []
    mse_train, mse_test = [], []
    r2_train, r2_test = [], []

    for degree in degrees_arr:
        # No intercept column: X_full's columns are x^1..x^max_degree, so the
        # first `degree` columns are x^1..x^degree.
        X_train = X_train_full[:, :degree]
        X_test = X_test_full[:, :degree]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        model = OLS().fit(X_train_s, y_train_centered)
        weights.append(model.coef_)

        y_pred_train = model.predict(X_train_s) + y_mean
        y_pred_test = model.predict(X_test_s) + y_mean

        mse_train.append(mean_squared_error(y_train, y_pred_train))
        mse_test.append(mean_squared_error(y_test, y_pred_test))
        r2_train.append(r2_score(y_train, y_pred_train))
        r2_test.append(r2_score(y_test, y_pred_test))

    return {
        "degrees": degrees_arr,
        "intercept": y_mean,
        "weights": weights,
        "mse_train": np.array(mse_train),
        "mse_test": np.array(mse_test),
        "r2_train": np.array(r2_train),
        "r2_test": np.array(r2_test),
    }

In [ ]:
# Baseline experiment: sweep polynomial degree at the original sample size (n=100).
degrees = range(1, 16)
results = fit_polynomial_ols_by_degree(x, y, degrees)

degrees_arr = results["degrees"]
intercept = results["intercept"]
weights = results["weights"]
mse_train, mse_test = results["mse_train"], results["mse_test"]
r2_train, r2_test = results["r2_train"], results["r2_test"]

## MSE and R2 vs. polynomial degree

Plot train/test MSE and R2 side by side to see the bias-variance trade-off as model complexity grows.

In [ ]:
degrees_arr = np.array(list(degrees))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(degrees_arr, mse_train, marker="o", markersize=3, label="train")
axes[0].plot(degrees_arr, mse_test, marker="o", markersize=3, label="test")
axes[0].set_yscale("log")
axes[0].set_xlabel("Polynomial degree")
axes[0].set_ylabel("MSE")
axes[0].set_title("MSE vs. polynomial degree")
axes[0].legend()

axes[1].plot(degrees_arr, r2_train, marker="o", markersize=3, label="train")
axes[1].plot(degrees_arr, r2_test, marker="o", markersize=3, label="test")
axes[1].axhline(0, color="gray", lw=0.5)
axes[1].set_xlabel("Polynomial degree")
axes[1].set_ylabel(r"$R^2$")
axes[1].set_title(r"$R^2$ vs. polynomial degree")
axes[1].legend()

fig.tight_layout()
plt.show()

## Coefficients ($\theta$) vs. polynomial degree

Each fit has `degree` coefficients: `theta[i]` is the coefficient of the
standardized $x^{i+1}$ term (there's no intercept column anymore — the intercept is
`y_mean`, a single scalar shared across all degrees, shown separately). Since
`weights` is a ragged list (more coefficients at higher degrees), pad it into a
rectangular matrix (missing entries as `NaN`) so each coefficient $\theta_i$ can be
traced as its own line across the degrees where it exists. This shows the intercept
staying constant by construction (it never depends on degree) while the high-order
coefficients blow up as the fit starts overfitting.

In [ ]:
max_degree = max(degrees)
theta_matrix = np.full((len(degrees), max_degree), np.nan)
for row, theta in enumerate(weights):
    theta_matrix[row, : len(theta)] = theta

fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(intercept, color="black", ls="--", lw=1, label=r"intercept ($\bar{y}_{train}$)")
for coef_idx in range(max_degree):
    label = rf"$\theta_{{{coef_idx + 1}}}$"
    ax.plot(
        degrees_arr,
        theta_matrix[:, coef_idx],
        marker="o",
        markersize=3,
        label=label,
    )

ax.set_yscale("symlog", linthresh=1)
ax.set_xlabel("Polynomial degree")
ax.set_ylabel(r"Coefficient value $\theta_i$")
ax.set_title(r"OLS coefficients $\theta_i$ vs. polynomial degree")
ax.legend(ncol=2, fontsize="small", loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.tight_layout()
plt.show()

## Effect of the number of data points

Part (a) also asks us to look at how training-set size affects the degree sweep,
not just the degree itself. We reuse `fit_polynomial_ols_by_degree` for a range of
dataset sizes $n \in \{50, 100, 300, 500, 1000\}$ — chosen to span a data-starved
regime ($n=30$, close to the 16 fitted parameters at the highest degree) up to a
comfortably over-determined one ($n=1000$) — and compare the test MSE and $R^2$
curves across degree for each $n$. All runs use the same noise level ($\sigma=0.1$)
and seed, so the only thing that changes between curves is the amount of data.

In [ ]:
# Re-run the same degree sweep for each dataset size, holding noise and seed fixed.
n_values = [50, 100, 300, 500, 1000]
noise_std = 0.1
seed = 42

n_sweep_results = {
    n: fit_polynomial_ols_by_degree(
        *generate_runge_data(n=n, noise_std=noise_std, seed=seed), degrees, seed=seed
    )
    for n in n_values
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for n in n_values:
    res = n_sweep_results[n]
    axes[0].plot(res["degrees"], res["mse_test"], marker="o", markersize=3, label=f"n={n}")
    axes[1].plot(res["degrees"], res["r2_test"], marker="o", markersize=3, label=f"n={n}")

axes[0].set_yscale("log")
axes[0].set_xlabel("Polynomial degree")
axes[0].set_ylabel("Test MSE")
axes[0].set_title("Test MSE vs. degree, by dataset size")
axes[0].legend(fontsize="small")

axes[1].axhline(0, color="gray", lw=0.5)
axes[1].set_xlabel("Polynomial degree")
axes[1].set_ylabel(r"Test $R^2$")
axes[1].set_title(r"Test $R^2$ vs. degree, by dataset size")
axes[1].legend(fontsize="small")

fig.tight_layout()
plt.show()

**Why does $n=100$ look *worse* than $n=50$ at low/mid degree, with a negative $R^2$?**

This is not a regression bug — the test *MSE* for $n=100$ is actually comparable to
or better than $n=50$ at almost every degree (e.g. degree 4: $0.025$ vs. $0.029$;
degree 10: $0.0094$ vs. $0.0058$). What differs is $R^2$, and that's an artifact of
evaluating on a small, fixed test split: $R^2 = 1 - SS_{res}/SS_{tot}$, and $SS_{tot}$
is just the variance of whatever `y_test` happens to contain.

The Runge function has a sharp peak at $x=0$ and is fairly flat elsewhere. With
`random_state=42`, the $n=50$ test split (10 points) happens to include one point
right next to the peak ($x=-0.067$, $y=0.915$), giving $\mathrm{Var}(y_{test})
\approx 0.056$. The $n=100$ test split (20 points) has no point closer than
$x=-0.258$ to the peak, so $\mathrm{Var}(y_{test}) \approx 0.023$ — less than half.
The same absolute test error therefore produces a much worse, even negative, $R^2$
for $n=100$, even though the underlying fit is not worse.

Takeaway: with a test set this small (10–20 points) and a target with localized
signal, $R^2$ from a single split is noisy and can be misleading — the test MSE
panel above is the more reliable basis for comparing across $n$.